# PitchLogic — Colab GPU 临时后端

**运行顺序：** Cell1 检查GPU → Cell2 装依赖 → Cell3 建目录 → Cell4 上传权重 → Cell5 克隆代码 → Cell6 写后端 → Cell7 启动 → Cell8 保活

In [8]:
# Cell 1: 检查 GPU
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️  No GPU — switch Runtime to T4/A100')


Fri Mar 27 07:41:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
# Cell 2: 安装依赖（首次约3-5分钟）
%pip install -q flask flask-cors ultralytics supervision \
    opencv-python-headless numpy pandas scikit-learn scipy \
    matplotlib decord seaborn Pillow loguru iopath tqdm lmdb \
    hydra-core jpeg4py \
    'git+https://github.com/roboflow/sports.git'

import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', '/usr/local/bin/cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ 依赖安装完成')


  Preparing metadata (setup.py) ... done
✅ 依赖安装完成


In [10]:
# Cell 3: 创建目录结构
import os
for d in [
    '/content/pitchlogic/backend/app/pipeline',
    '/content/pitchlogic/backend/app/routes',
    '/content/pitchlogic/backend/weights/football',
    '/content/pitchlogic/backend/weights/keypoints',
    '/content/pitchlogic/outputs',
    '/content/pitchlogic/uploads',
]:
    os.makedirs(d, exist_ok=True)
print('✅ 目录结构创建完成')


✅ 目录结构创建完成


In [11]:
# Cell 4: 挂载 Drive 并复制模型权重
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
shutil.copy(
    '/content/drive/MyDrive/samurai_env/samurai/weight_football/best.pt',
    '/content/pitchlogic/backend/weights/football/best.pt'
)
shutil.copy(
    '/content/drive/MyDrive/samurai_env/samurai/weights_keypoints/best.pt',
    '/content/pitchlogic/backend/weights/keypoints/best.pt'
)

fb = os.path.exists('/content/pitchlogic/backend/weights/football/best.pt')
kb = os.path.exists('/content/pitchlogic/backend/weights/keypoints/best.pt')
print('Football model:', '✅' if fb else '❌ 缺失')
print('Keypoint model:', '✅' if kb else '❌ 缺失')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Football model: ✅
Keypoint model: ✅


In [12]:
# Cell 5: 从 GitHub 克隆/更新后端代码
import subprocess, shutil, os

REPO_DIR = '/content/AI_football_analizer'

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Muriki1234/AI_football_analizer.git', REPO_DIR],
        check=True
    )
    print('✅ 仓库克隆完成')
else:
    # 强制拉取最新代码
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
    print('✅ 仓库已更新到最新')

# 复制 backend/app 和 config.py
shutil.copytree(f'{REPO_DIR}/backend/app',
                '/content/pitchlogic/backend/app',
                dirs_exist_ok=True)
shutil.copy(f'{REPO_DIR}/backend/config.py',
            '/content/pitchlogic/backend/config.py')

# 验证关键文件
for f in [
    'app/pipeline/session_manager.py',
    'app/pipeline/tasks.py',
    'app/pipeline/analysis_core.py',
    'config.py',
]:
    ok = os.path.exists(f'/content/pitchlogic/backend/{f}')
    print(f'{"✅" if ok else "❌"} {f}')


仓库已存在，跳过克隆
✅ app/pipeline/session_manager.py
✅ app/pipeline/tasks.py
✅ app/pipeline/analysis_core.py
✅ config.py


In [13]:
# Cell 6: 从仓库复制后端文件（无需手动改代码，改代码只需 push 到 GitHub）
import shutil
shutil.copy('/content/AI_football_analizer/colab_backend.py', '/content/colab_backend.py')
print('✅ colab_backend.py 已从 GitHub 仓库复制')


In [14]:
# Cell 7: 启动 Flask + cloudflared 隧道
import threading, time, os, requests, subprocess, re

def run_flask():
    os.system('python /content/colab_backend.py')

threading.Thread(target=run_flask, daemon=True).start()

# 等待 Flask 启动
public_url = None
for i in range(10):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:7860/health', timeout=3)
        print(f'✅ Flask OK: {r.json()["gpu"]["name"]}')
        break
    except:
        print(f'⏳ 等待 Flask... ({(i+1)*2}s)')

# 启动隧道
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
print('⏳ 等待隧道...')
for _ in range(60):
    line = tunnel.stdout.readline()
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print()
print('='*60)
print('🎉 COLAB 后端已就绪！')
print(f'URL: {public_url}')
print('='*60)
print(f'健康检查: {public_url}/health')




⏳ 等待 Flask... (2s)
⏳ 等待 Flask... (4s)
⏳ 等待 Flask... (6s)
⏳ 等待 Flask... (8s)
⏳ 等待 Flask... (10s)
⏳ 等待 Flask... (12s)
⏳ 等待 Flask... (14s)
⏳ 等待 Flask... (16s)
⏳ 等待 Flask... (18s)
✅ Flask OK: Tesla T4
⏳ 等待隧道...

🎉 COLAB 后端已就绪！
URL: https://district-aspect-jones-man.trycloudflare.com
健康检查: https://district-aspect-jones-man.trycloudflare.com/health


In [ ]:
# Cell 8: 保活（防止 Colab 自动断开）
import requests, threading, time

def heartbeat():
    while True:
        try:
            r = requests.get('http://localhost:7860/health', timeout=5)
            name = r.json().get('gpu', {}).get('name', '?')
            print(f'💓 heartbeat OK — {name}', flush=True)
        except Exception as e:
            print(f'⚠️ heartbeat failed: {e}', flush=True)
        time.sleep(300)

threading.Thread(target=heartbeat, daemon=True).start()
print('✅ 保活已启动（5分钟心跳）')
print()
print('浏览器端保活，复制到 DevTools Console：')
print('setInterval(()=>document.dispatchEvent(new MouseEvent("mousemove",{bubbles:true})),60000)')


In [ ]:
 !ls -d /content/drive/MyDrive/samurai_env/samurai/sam2

/content/drive/MyDrive/samurai_env/samurai/sam2
